# 01 · Exploratory analysis — PJM East hourly load

Four questions, in the order that a forecasting model needs them answered:

1. **What cycles does the series have?** → decides the calendar features.
2. **Is there a trend under the seasonality?** → decides whether levels can be modelled directly.
3. **What drives the level besides the clock?** → decides the weather features.
4. **How far back does memory reach?** → decides the lag set.

Every figure is produced by `src.report`, not by code in this notebook — the notebook is a
presentation surface, so the same function generates the README image and the cell output.

In [ ]:
import sys

sys.path.insert(0, "..")

import pandas as pd
from IPython.display import Image, display

from src import report, viz
from src.config import CLEAN_PARQUET, LOAD_COL, LOCAL_TZ

viz.apply_theme()
clean = pd.read_parquet(CLEAN_PARQUET)
clean.head()

## The cleaned series

The index is a continuous hourly UTC grid — no gaps, no duplicates, no DST artefacts.
The flag columns record what the cleaning pass had to do to get there.

In [ ]:
local = clean.index.tz_convert(LOCAL_TZ)
print(f"rows        : {len(clean):,}")
print(f"span (UTC)  : {clean.index.min()} .. {clean.index.max()}")
print(f"span (local): {local.min()} .. {local.max()}")
print(f"years       : {(clean.index.max() - clean.index.min()).days / 365.25:.1f}")
print()
print(clean[[LOAD_COL, "temp_c"]].describe().T.to_string(float_format=lambda v: f"{v:,.1f}"))
print()
print("cleaning flags (share of rows):")
print((clean[["imputed", "clipped", "bad_day", "temp_imputed"]].mean() * 100).round(3).to_string())

## 1 · Three nested cycles

The daily profile has the classic double hump (a morning ramp, an evening peak). Weekends sit
several thousand MW below weekdays at the same hour and the peak is flatter — commercial and
industrial demand is what goes missing. The heatmap shows both cycles at once: Monday–Friday
read as one block, Saturday and Sunday as another.

**Feature consequence:** hour-of-day and day-of-week both matter, and both are cyclical —
hour 23 is adjacent to hour 0 — so they are encoded as sin/cos pairs rather than integers.

In [ ]:
display(Image(report.plot_seasonality(clean)))

## 2 · STL: what is left once the annual cycle is removed

STL runs on the daily mean rather than the hourly series: at hourly resolution the annual
period is 8,766 observations, which is both slow to fit and impossible to read.

**Feature consequence:** if the trend is close to flat, the level can be modelled directly
without differencing or detrending — which is why the ARIMA component below is specified
with `d=0` and the seasonality handled by exogenous Fourier terms instead.

In [ ]:
display(Image(report.plot_stl(clean)))

## 3 · The temperature U

Load against temperature is not monotone — it is a U. Demand rises to the left of the
comfort band (electric heating) and rises far more steeply to the right (air conditioning).
A single linear temperature term would average those two arms into nothing.

**Feature consequence:** split the response into two one-sided hinges,
`hdd = max(18 − T, 0)` and `cdd = max(T − 24, 0)`, so each arm gets its own slope.

**Honesty note:** these features use the *realised* temperature at the target hour, i.e. a
perfect weather forecast. A production system would substitute a real forecast and lose some
accuracy; the assumption is stated wherever these results are reported.

In [ ]:
display(Image(report.plot_load_vs_temperature(clean)))

## 4 · How far back memory reaches

The autocorrelation function does not decay smoothly — it peaks at multiples of 24 hours, and
the 168-hour peak (same hour, same weekday, last week) stands above its neighbours.

**Feature consequence:** the lag set is `lag_24`, `lag_48`, `lag_168` — chosen from these
peaks rather than from a grid search, and `lag_168` is exactly the seasonal-naive baseline
that every model is scored against.

In [ ]:
display(Image(report.plot_autocorrelation(clean)))

## What the EDA settles

| Observation | Consequence for the model |
|---|---|
| Daily + weekly + annual cycles | cyclical hour/day-of-week/month encodings; US holiday flag |
| Trend is weak relative to seasonality | model the level directly (`d=0`), no detrending step |
| Load vs temperature is U-shaped | two one-sided hinges (HDD/CDD), not a linear temperature term |
| ACF peaks at 24h and 168h | `lag_24`, `lag_48`, `lag_168`; seasonal naive at 168h is the baseline |

Next: `02_results.ipynb` — the backtest protocol and the model comparison.